<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/NLP-2026/Lecture_2/GloVe_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GloVe: Global Vectors for Word Representation

## Подробная теория с полными выводами и объяснениями

---

## Введение: почему GloVe заслуживает отдельного разбора

К моменту появления GloVe в 2014 году в арсенале исследователей уже были два подхода к обучению эмбеддингов: матричные методы (LSA) и нейросетевые методы (Word2Vec). Каждый из них имел свои сильные и слабые стороны, и каждый оставлял нерешённые проблемы. GloVe, предложенный Джеффри Пеннингтоном, Ричардом Сочером и Кристофером Мэннингом, попытался объединить лучшее из обоих миров.

Чтобы понять, почему GloVe важен, нужно сначала понять, что именно не устраивало исследователей в существующих методах.

**Проблема LSA.** Латентный семантический анализ работает с глобальной статистикой: он строит матрицу «термин-документ» и раскладывает её с помощью сингулярного разложения. Это даёт плотные векторы, но LSA не имеет вероятностной интерпретации. Его латентные факторы — это просто направления в пространстве, которые трудно объяснить содержательно. Кроме того, LSA плохо масштабируется: SVD для матрицы размера $10^6 \times 10^6$ вычислительно невозможен.

**Проблема Word2Vec.** Word2Vec обучается на локальных парах слов в скользящем окне. Он предсказывает контекст и постепенно настраивает векторы. Это даёт плотные семантически насыщенные векторы, но Word2Vec не использует глобальную статистику явно. Информация о том, что «кошка» и «собака» часто встречаются с «сидит», размазана по миллионам локальных обновлений. Кроме того, Word2Vec требует тщательной настройки гиперпараметров (размер окна, количество отрицательных примеров, субсэмплирование) и может быть нестабильным.

**Идея GloVe.** А что если мы объединим глобальную статистику LSA с вероятностной моделью Word2Vec? Что если мы построим матрицу совместной встречаемости слов (как в LSA), но обучим векторы через вероятностную модель (как в Word2Vec)? Именно это и делает GloVe. Он строит матрицу совместной встречаемости $X$, где $X_{ij}$ — число раз, когда слово $j$ встречается в контексте слова $i$, а затем обучает векторы так, чтобы их скалярное произведение воспроизводило логарифм $X_{ij}$.

Ключевое преимущество GloVe: он использует **всю** глобальную статистику одновременно, а не обрабатывает пары по одной. Это делает обучение более стабильным и позволяет лучше работать с редкими словами. Кроме того, GloVe имеет чёткую вероятностную интерпретацию: скалярное произведение векторов кодирует логарифм вероятности совместной встречаемости.

В этой лекции мы подробно разберём:

- что такое матрица совместной встречаемости и как она строится;
- ключевую идею GloVe — отношения вероятностей;
- полный вывод функциональной формы;
- вывод функции потерь через вероятностную модель;
- взвешенную логистическую регрессию и её свойства;
- связь с матричной факторизацией;
- градиенты и алгоритм обучения;
- практические детали и гиперпараметры;
- сравнение с Word2Vec;
- ограничения и расширения.

Мы будем следовать той же структуре, что и в лекциях по Word2Vec, но с акцентом на особенности GloVe.

---

## 1. Матрица совместной встречаемости

### 1.1 Определение и построение

Пусть дан корпус — последовательность слов $w_1, w_2, \ldots, w_T$, где $T$ — длина корпуса. Словарь $V = \{w_1, \ldots, w_N\}$, где $N = |V|$ — размер словаря.

**Матрица совместной встречаемости** $X \in \mathbb{R}^{N \times N}$ определяется так:

$$
X_{ij} = \text{число раз, когда слово } j \text{ встречается в контексте слова } i.
$$

Что означает «в контексте»? Зафиксируем размер окна $m$. Для каждого вхождения слова $i$ в позиции $t$ мы смотрим на слова в позициях $t-m, \ldots, t-1, t+1, \ldots, t+m$. Каждое такое вхождение увеличивает $X_{ij}$ на 1.

**Пример.** Рассмотрим корпус «кошка сидит на окне», окно $m = 1$. Последовательность слов: кошка, сидит, на, окне.

- Позиция 1 (кошка): контекст — «сидит». $X_{\text{кошка}, \text{сидит}} += 1$.
- Позиция 2 (сидит): контекст — «кошка», «на». $X_{\text{сидит}, \text{кошка}} += 1$, $X_{\text{сидит}, \text{на}} += 1$.
- Позиция 3 (на): контекст — «сидит», «окне». $X_{\text{на}, \text{сидит}} += 1$, $X_{\text{на}, \text{окне}} += 1$.
- Позиция 4 (окне): контекст — «на». $X_{\text{окне}, \text{на}} += 1$.

Получаем матрицу:

| | кошка | сидит | на | окне |
|---|---|---|---|---|
| кошка | 0 | 1 | 0 | 0 |
| сидит | 1 | 0 | 1 | 0 |
| на | 0 | 1 | 0 | 1 |
| окне | 0 | 0 | 1 | 0 |

**Тонкий момент.** Матрица $X$ **не симметрична** в общем случае. $X_{ij}$ — это число раз, когда $j$ встречается в контексте $i$. Но $X_{ji}$ — это число раз, когда $i$ встречается в контексте $j$. Эти числа могут различаться, потому что слова имеют разные частоты. В нашем примере $X_{\text{кошка}, \text{сидит}} = 1$, но $X_{\text{сидит}, \text{кошка}} = 1$ — совпало. Однако если бы слово «сидит» встречалось чаще, $X_{\text{сидит}, \text{кошка}}$ было бы больше.

На практике часто используют **симметричную** версию: $X_{ij} \leftarrow X_{ij} + X_{ji}$, или просто работают с несимметричной матрицей.

### 1.2 Свойства матрицы

**Размерность.** $N \times N$, где $N$ — размер словаря. Для $N = 10^6$ это $10^{12}$ элементов. Хранить такую матрицу в памяти невозможно. Но матрица **разрежена**: большинство пар слов никогда не встречаются вместе. Число ненулевых элементов пропорционально $T \times m$, где $T$ — длина корпуса. Для корпуса из миллиарда слов и окна $m = 10$ это $10^{10}$ ненулевых элементов — всё ещё много, но управляемо с помощью разреженных структур.

**Частоты.** Сумма по строке $i$:

$$
X_i = \sum_{j=1}^{N} X_{ij}
$$

— это общее число раз, когда слово $i$ встречалось в позиции целевого слова, умноженное на $2m$ (потому что для каждого вхождения мы смотрим $2m$ контекстных слов). Точнее, $X_i = 2m \cdot \text{count}(w_i)$, где $\text{count}(w_i)$ — частота слова $i$ в корпусе (если не учитывать границы).

**Вероятности.** Вероятность того, что слово $j$ встречается в контексте слова $i$:

$$
P_{ij} = P(j \mid i) = \frac{X_{ij}}{X_i}.
$$

Это ключевая величина для GloVe. Она показывает, насколько слово $j$ характерно для контекста слова $i$.

### 1.3 Почему матрица совместной встречаемости важна

Матрица $X$ содержит **глобальную статистику** совместной встречаемости всех пар слов в корпусе. Она агрегирует информацию из миллионов локальных контекстов. Если два слова часто встречаются вместе, $X_{ij}$ велико. Если они никогда не встречаются вместе, $X_{ij} = 0$.

**Ключевое наблюдение.** Матрица $X$ содержит всю информацию, необходимую для изучения семантики. Действительно, дистрибутивная гипотеза утверждает, что значение слова определяется его контекстами. А контексты — это именно то, что записано в матрице $X$. Строка $i$ матрицы $X$ — это распределение слова $i$ по контекстам. Если две строки похожи, слова семантически близки.

**Идея GloVe.** Мы хотим найти векторы $u_w, v_w \in \mathbb{R}^d$ такие, что их скалярное произведение $u_i^\top v_j$ **воспроизводит** статистику совместной встречаемости. Иными словами, мы хотим, чтобы $u_i^\top v_j$ было большим, когда $X_{ij}$ велико, и маленьким, когда $X_{ij}$ мало.

Но как именно связать $u_i^\top v_j$ с $X_{ij}$? Ответ на этот вопрос — ключевой вклад GloVe.

---

## 2. Ключевая идея: отношения вероятностей

### 2.1 Интуиция через пример

Рассмотрим три слова: $i$ = «лёд», $j$ = «пар», $k$ = «твёрдый». Мы хотим понять, чем «лёд» отличается от «пара».

- «Лёд» встречается с «твёрдый» часто, «пар» — редко. Отношение $P(\text{твёрдый} \mid \text{лёд}) / P(\text{твёрдый} \mid \text{пар})$ будет большим.
- «Лёд» встречается с «горячий» редко, «пар» — часто. Отношение $P(\text{горячий} \mid \text{лёд}) / P(\text{горячий} \mid \text{пар})$ будет маленьким.
- «Лёд» и «пар» оба встречаются с «вода» часто. Отношение $P(\text{вода} \mid \text{лёд}) / P(\text{вода} \mid \text{пар})$ будет близко к 1.
- «Лёд» и «пар» оба редко встречаются с «мобильный». Отношение $P(\text{мобильный} \mid \text{лёд}) / P(\text{мобильный} \mid \text{пар})$ будет близко к 1.

**Ключевое наблюдение.** Отношение вероятностей $P_{ik} / P_{jk}$ несёт информацию о связи слов $i$ и $j$ через контекст $k$. Это отношение может быть:

- большим, если $k$ связан с $i$, но не с $j$;
- маленьким, если $k$ связан с $j$, но не с $i$;
- близким к 1, если $k$ связан с обоими или ни с одним.

Именно это отношение, а не сами вероятности, позволяет различать слова. Вероятности $P_{ik}$ и $P_{jk}$ могут быть обе большими или обе маленькими, но их отношение — информативно.

### 2.2 Формализация

Пусть $P_{ij} = P(j \mid i) = X_{ij} / X_i$. Рассмотрим отношение:

$$
R_{ijk} = \frac{P_{ik}}{P_{jk}}.
$$

Мы хотим, чтобы это отношение **выражалось через векторы** слов. Пусть $u_i, u_j, u_k \in \mathbb{R}^d$ — векторы слов. Мы предполагаем, что существует функция $F$ такая, что:

$$
F(u_i, u_j, u_k) = \frac{P_{ik}}{P_{jk}}.
$$

Наша задача — найти $F$ и векторы, которые удовлетворяют этому уравнению.

### 2.3 Вывод функциональной формы

Мы хотим, чтобы $F$ зависела от разности $u_i - u_j$ (потому что отношение $P_{ik}/P_{jk}$ симметрично относительно замены $i \leftrightarrow j$ с точностью до обращения). Также $F$ должна зависеть от $u_k$ (потому что контекст $k$ определяет, какое именно отношение мы рассматриваем).

Простейшая форма:

$$
F(u_i - u_j, u_k) = \frac{P_{ik}}{P_{jk}}.
$$

Мы хотим, чтобы $F$ была **гомоморфизмом** между группой векторов (с операцией сложения) и группой положительных чисел (с операцией умножения). Это означает, что $F$ должна быть экспоненциальной функцией. Действительно:

$$
\exp(a + b) = \exp(a) \cdot \exp(b).
$$

Поэтому естественно предположить:

$$
F(u_i - u_j, u_k) = \exp\left( (u_i - u_j)^\top u_k \right).
$$

Тогда:

$$
\exp\left( (u_i - u_j)^\top u_k \right) = \frac{P_{ik}}{P_{jk}}.
$$

Возьмём логарифм:

$$
(u_i - u_j)^\top u_k = \log P_{ik} - \log P_{jk}.
$$

Это уравнение должно выполняться для всех $i, j, k$. Перепишем его в виде:

$$
u_i^\top u_k - \log P_{ik} = u_j^\top u_k - \log P_{jk}.
$$

Левая часть зависит только от $i$ и $k$, правая — только от $j$ и $k$. Но равенство должно выполняться для всех $i, j$. Это возможно только если обе части равны некоторой функции, зависящей только от $k$:

$$
u_i^\top u_k - \log P_{ik} = b_k,
$$

где $b_k$ — функция от $k$. Аналогично, можно ввести $b_i$ как функцию от $i$. Тогда:

$$
u_i^\top u_k = \log P_{ik} + b_i + b_k.
$$

**Тонкий момент.** Мы получили уравнение, которое связывает скалярное произведение векторов с логарифмом вероятности совместной встречаемости. Это **ключевое уравнение GloVe**. Оно говорит, что скалярное произведение векторов двух слов должно быть равно логарифму их совместной встречаемости плюс смещения, которые компенсируют разную частоту слов.

### 2.4 Симметризация

Уравнение $u_i^\top u_k = \log P_{ik} + b_i + b_k$ не симметрично: $P_{ik}$ и $P_{ki}$ могут различаться. Чтобы сделать модель симметричной, введём два набора векторов: $u_i$ (для слова как целевого) и $v_j$ (для слова как контекстного). Тогда:

$$
u_i^\top v_j = \log X_{ij} + b_i + \tilde{b}_j,
$$

где $b_i$ и $\tilde{b}_j$ — смещения (biases) для слова $i$ и слова $j$.

Здесь мы заменили $\log P_{ij}$ на $\log X_{ij}$, потому что $P_{ij} = X_{ij} / X_i$, а $\log X_i$ можно поглотить в $b_i$.

**Интерпретация.** Скалярное произведение $u_i^\top v_j$ должно быть равно логарифму числа совместных вхождений слова $i$ и слова $j$, плюс смещения. Смещения компенсируют разную частоту слов: частые слова имеют большие $X_i$, что отражается в $b_i$.

---

## 3. Функция потерь GloVe

### 3.1 Наивная функция потерь

Уравнение $u_i^\top v_j + b_i + \tilde{b}_j = \log X_{ij}$ мотивирует следующую функцию потерь:

$$
J = \sum_{i,j=1}^{N} \left( u_i^\top v_j + b_i + \tilde{b}_j - \log X_{ij} \right)^2.
$$

Это метод наименьших квадратов: мы минимизируем сумму квадратов разностей между скалярным произведением (плюс смещения) и логарифмом совместной встречаемости.

**Проблема.** Эта функция потерь имеет два серьёзных недостатка.

1. **Неопределённость при $X_{ij} = 0$.** Если слова $i$ и $j$ никогда не встречаются вместе, $X_{ij} = 0$, и $\log X_{ij} = -\infty$. Функция потерь становится неопределённой.

2. **Дисбаланс частот.** Большинство пар слов имеют $X_{ij} = 0$ или очень маленькое $X_{ij}$. Если мы просто игнорируем нулевые пары, мы теряем информацию о том, что эти слова **не** встречаются вместе. Если мы включаем их с весом 1, они доминируют в функции потерь (потому что их большинство), и модель не учится на редких, но важных парах.

### 3.2 Взвешенная функция потерь

GloVe решает эти проблемы, вводя **весовую функцию** $f(X_{ij})$:

$$
J = \sum_{i,j=1}^{N} f(X_{ij}) \left( u_i^\top v_j + b_i + \tilde{b}_j - \log X_{ij} \right)^2.
$$

**Свойства $f$:**

1. $f(0) = 0$. Если $X_{ij} = 0$, пара игнорируется. Это решает проблему $\log 0$.
2. $f(x)$ должна быть **неубывающей**: редкие пары получают меньший вес, частые — больший.
3. $f(x)$ должна **насыщаться** при больших $x$: очень частые пары не должны доминировать. Иначе модель будет тратить всё время на частые слова (артикли, предлоги) и плохо обучать редкие.

**Выбор $f$:**

$$
f(x) = \begin{cases} (x / x_{\max})^\alpha, & \text{если } x < x_{\max}, \\ 1, & \text{если } x \ge x_{\max}. \end{cases}
$$

Параметры:

- $x_{\max} = 100$ (типичное значение);
- $\alpha = 3/4$ (типичное значение).

**Анализ.**

- При $x \to 0$: $f(x) \to 0$. Редкие пары игнорируются.
- При $x = x_{\max}$: $f(x) = 1$. Достигается максимум.
- При $x > x_{\max}$: $f(x) = 1$. Насыщение.

**Почему $\alpha = 3/4$?** Это эмпирический выбор. При $\alpha = 1$ функция линейна, и частые пары доминируют. При $\alpha = 0$ функция постоянна (кроме $x = 0$), и все пары имеют одинаковый вес. Степень $3/4$ — компромисс, который хорошо работает на практике. Тот же показатель используется в negative sampling Word2Vec.

**Почему $x_{\max} = 100$?** Это также эмпирический выбор. При $x_{\max}$ слишком большом насыщение не достигается, и частые пары доминируют. При слишком маленьком — теряется информация о частых парах. Значение 100 хорошо работает для большинства корпусов.

### 3.3 Свойства функции потерь

**Невыпуклость.** Функция $J$ невыпукла по $u_i, v_j, b_i, \tilde{b}_j$ из-за произведения $u_i^\top v_j$. Это означает, что глобальный минимум не гарантирован. На практике используется стохастический градиентный спуск (SGD) или AdaGrad, которые сходятся к локальному минимуму.

**Симметрия.** Если поменять местами $u_i$ и $v_i$, а также $b_i$ и $\tilde{b}_i$, значение $J$ не изменится (при симметричной $f$ и симметричной матрице $X$). Это свойство идентифицируемости: решение не единственно. После обучения обычно используют $u_i + v_i$ как итоговый эмбеддинг.

**Смещения.** Смещения $b_i$ и $\tilde{b}_j$ играют важную роль: они компенсируют разную частоту слов. Без них модель была бы вынуждена кодировать частоту в векторах, что ухудшило бы качество.

### 3.4 Связь с матричной факторизацией

Функция потерь GloVe:

$$
J = \sum_{i,j} f(X_{ij}) \left( u_i^\top v_j + b_i + \tilde{b}_j - \log X_{ij} \right)^2
$$

— это **взвешенная матричная факторизация**. Мы аппроксимируем матрицу $Y_{ij} = \log X_{ij}$ произведением $U V^\top$ плюс смещения.

**Отличие от LSA.** LSA использует SVD для факторизации матрицы $X$ (или TF-IDF), минимизируя ошибку в смысле Фробениуса. GloVe минимизирует **взвешенную** ошибку, где вес $f(X_{ij})$ зависит от частоты пары. Это позволяет GloVe лучше работать с редкими парами и игнорировать нулевые.

**Отличие от Word2Vec.** Word2Vec обучается на локальных парах, GloVe — на глобальной матрице. Word2Vec использует negative sampling, GloVe — взвешенную регрессию. Оба подхода дают похожие результаты, но разными путями.

---

## 4. Вывод функции потерь через вероятностную модель

### 4.1 Вероятностная интерпретация

Функцию потерь GloVe можно вывести из вероятностной модели. Предположим, что:

$$
X_{ij} \sim \text{Poisson}\left( \exp(u_i^\top v_j + b_i + \tilde{b}_j) \right).
$$

Тогда логарифм правдоподобия:

$$
\log P(X_{ij} \mid u_i, v_j) = X_{ij} (u_i^\top v_j + b_i + \tilde{b}_j) - \exp(u_i^\top v_j + b_i + \tilde{b}_j) - \log X_{ij}!.
$$

Максимизация этого правдоподобия эквивалентна минимизации:

$$
\exp(u_i^\top v_j + b_i + \tilde{b}_j) - X_{ij} (u_i^\top v_j + b_i + \tilde{b}_j).
$$

Это не то же самое, что функция потерь GloVe. Однако если предположить, что $X_{ij}$ — это **логарифмически нормальная** величина, мы получим квадратичную функцию потерь.

### 4.2 Вывод через логарифмически нормальную модель

Предположим:

$$
\log X_{ij} = u_i^\top v_j + b_i + \tilde{b}_j + \epsilon_{ij},
$$

где $\epsilon_{ij} \sim \mathcal{N}(0, \sigma^2)$ — гауссовский шум. Тогда:

$$
P(\log X_{ij} \mid u_i, v_j) \propto \exp\left( -\frac{(\log X_{ij} - u_i^\top v_j - b_i - \tilde{b}_j)^2}{2\sigma^2} \right).
$$

Логарифм правдоподобия для всей матрицы:

$$
\log \mathcal{L} = -\sum_{i,j} \frac{(\log X_{ij} - u_i^\top v_j - b_i - \tilde{b}_j)^2}{2\sigma^2} + \text{const}.
$$

Максимизация $\log \mathcal{L}$ эквивалентна минимизации:

$$
J = \sum_{i,j} \left( \log X_{ij} - u_i^\top v_j - b_i - \tilde{b}_j \right)^2.
$$

Это **невзвешенная** функция потерь. Чтобы получить взвешенную, добавим вес $f(X_{ij})$:

$$
J = \sum_{i,j} f(X_{ij}) \left( \log X_{ij} - u_i^\top v_j - b_i - \tilde{b}_j \right)^2.
$$

**Интерпретация веса.** $f(X_{ij})$ можно рассматривать как **точность** наблюдения. Чем чаще пара встречается, тем точнее мы знаем её статистику, тем больше вес. Но при очень больших $X_{ij}$ вес насыщается, чтобы не переобучаться на частых парах.

### 4.3 Почему не Poisson?

Можно было бы использовать Poisson-модель:

$$
X_{ij} \sim \text{Poisson}\left( \exp(u_i^\top v_j + b_i + \tilde{b}_j) \right).
$$

Тогда логарифм правдоподобия:

$$
\log P(X_{ij}) = X_{ij} (u_i^\top v_j + b_i + \tilde{b}_j) - \exp(u_i^\top v_j + b_i + \tilde{b}_j) - \log X_{ij}!.
$$

Максимизация этого правдоподобия эквивалентна минимизации:

$$
J_{\text{Poisson}} = \sum_{i,j} \left[ \exp(u_i^\top v_j + b_i + \tilde{b}_j) - X_{ij} (u_i^\top v_j + b_i + \tilde{b}_j) \right].
$$

Эта функция потерь имеет свои преимущества (она выпукла по $u_i^\top v_j$), но она вычислительно сложнее и не так хорошо работает на практике, как взвешенная квадратичная. GloVe использует квадратичную функцию, потому что она проще и даёт хорошие результаты.

---

## 5. Градиенты и обучение

### 5.1 Обозначения

Для пары $(i, j)$:

- $u_i \in \mathbb{R}^d$ — вектор целевого слова;
- $v_j \in \mathbb{R}^d$ — вектор контекстного слова;
- $b_i \in \mathbb{R}$ — смещение целевого слова;
- $\tilde{b}_j \in \mathbb{R}$ — смещение контекстного слова;
- $X_{ij}$ — число совместных вхождений;
- $f_{ij} = f(X_{ij})$ — вес.

**Ошибка:**

$$
e_{ij} = u_i^\top v_j + b_i + \tilde{b}_j - \log X_{ij}.
$$

**Функция потерь для одной пары:**

$$
J_{ij} = f_{ij} \cdot e_{ij}^2.
$$

### 5.2 Градиенты

**По $u_i$:**

$$
\frac{\partial J_{ij}}{\partial u_i} = 2 f_{ij} e_{ij} v_j.
$$

**По $v_j$:**

$$
\frac{\partial J_{ij}}{\partial v_j} = 2 f_{ij} e_{ij} u_i.
$$

**По $b_i$:**

$$
\frac{\partial J_{ij}}{\partial b_i} = 2 f_{ij} e_{ij}.
$$

**По $\tilde{b}_j$:**

$$
\frac{\partial J_{ij}}{\partial \tilde{b}_j} = 2 f_{ij} e_{ij}.
$$

### 5.3 Обновление параметров

Используя градиентный спуск (мы минимизируем $J$):

$$
u_i \leftarrow u_i - \eta \cdot 2 f_{ij} e_{ij} v_j,
$$

$$
v_j \leftarrow v_j - \eta \cdot 2 f_{ij} e_{ij} u_i,
$$

$$
b_i \leftarrow b_i - \eta \cdot 2 f_{ij} e_{ij},
$$

$$
\tilde{b}_j \leftarrow \tilde{b}_j - \eta \cdot 2 f_{ij} e_{ij},
$$

где $\eta$ — скорость обучения.

### 5.4 AdaGrad

На практике GloVe использует **AdaGrad** вместо обычного SGD. AdaGrad адаптирует скорость обучения для каждого параметра:

$$
u_i \leftarrow u_i - \frac{\eta}{\sqrt{G_{u_i}}} \cdot 2 f_{ij} e_{ij} v_j,
$$

где $G_{u_i}$ — накопленная сумма квадратов градиентов по $u_i$:

$$
G_{u_i} \leftarrow G_{u_i} + (2 f_{ij} e_{ij} v_j)^2.
$$

**Преимущество AdaGrad.** Редкие слова получают большие эффективные скорости обучения, потому что их градиенты малы. Это помогает обучать редкие слова.

**Тонкий момент.** В GloVe AdaGrad инициализируется значением 1 для всех параметров, а не 0, чтобы избежать деления на ноль.

### 5.5 Полный алгоритм

1. **Построить матрицу совместной встречаемости** $X$ на корпусе.
2. **Инициализировать** $U, V, b, \tilde{b}$ случайными малыми значениями.
3. **Для каждой эпохи:**
   - Для каждой пары $(i, j)$ с $X_{ij} > 0$:
     - Вычислить $e_{ij} = u_i^\top v_j + b_i + \tilde{b}_j - \log X_{ij}$.
     - Вычислить $f_{ij} = f(X_{ij})$.
     - Обновить параметры по формулам AdaGrad.
4. **Повторять** до сходимости.

**Тонкий момент.** На практике пары $(i, j)$ сэмплируются пропорционально $X_{ij}$ или равномерно. В оригинальной реализации используется случайный порядок.

### 5.6 Выбор гиперпараметров

- $d = 50$–$300$: размерность эмбеддинга.
- $x_{\max} = 100$: порог насыщения веса.
- $\alpha = 3/4$: показатель степени веса.
- $\eta = 0.05$: начальная скорость обучения.
- Эпох: 25–50.
- Контекстное окно: $m = 10$ (или динамическое).

---

## 6. Связь с Word2Vec

### 6.1 Общие черты

- Оба метода дают плотные векторы размерности $d = 100$–$300$.
- Оба используют скалярное произведение векторов как меру совместимости.
- Оба используют взвешивание, чтобы уменьшить влияние частых слов (степень $3/4$).
- Оба дают похожие результаты на бенчмарках (аналогии, близость слов).

### 6.2 Различия

| Свойство | Word2Vec | GloVe |
|----------|----------|-------|
| Данные | Локальные пары | Глобальная матрица |
| Обучение | Предсказание контекста | Взвешенная регрессия |
| Оптимизация | SGD / negative sampling | AdaGrad |
| Функция потерь | Логистическая | Квадратичная |
| Частые слова | Negative sampling | Вес $f(X_{ij})$ |
| Редкие слова | Skip-gram лучше | Вес насыщается |
| Скорость | Быстрее на больших корпусах | Медленнее, но точнее |
| Память | Меньше | Больше (матрица $X$) |

### 6.3 Когда что использовать

- **Word2Vec:** большие корпуса, ограниченное время, потоковые данные.
- **GloVe:** средние корпуса, важна точность, есть возможность построить матрицу $X$.
- **На практике:** оба метода дают похожие результаты. Выбор часто определяется удобством реализации.

---

## 7. Свойства эмбеддингов GloVe

### 7.1 Линейные аналогии

GloVe, как и Word2Vec, поддерживает линейные аналогии:

$$
u_{\text{король}} - u_{\text{мужчина}} + u_{\text{женщина}} \approx u_{\text{королева}}.
$$

Это свойство возникает потому, что скалярное произведение $u_i^\top v_j$ кодирует отношения между словами. Если «король» и «королева» различаются только по признаку пола, а «мужчина» и «женщина» — тоже, то разность векторов должна быть одинаковой.

### 7.2 Косинусная близость

Косинусная близость между векторами:

$$
\text{sim}(i, j) = \frac{u_i^\top u_j}{\|u_i\| \|u_j\|}.
$$

Семантически близкие слова имеют высокую косинусную близость.

### 7.3 Нормализация

После обучения векторы часто нормализуют по L2:

$$
u_i \leftarrow \frac{u_i}{\|u_i\|_2}.
$$

Это делает косинусную близость эквивалентной скалярному произведению.

### 7.4 Смещения

Смещения $b_i$ и $\tilde{b}_j$ кодируют частоту слов. Слова с высокой частотой имеют большие смещения. Это позволяет векторам кодировать семантику, а не частоту.

---

## 8. Ограничения GloVe

### 8.1 Полисемия

Как и Word2Vec, GloVe даёт **один вектор на слово**, независимо от контекста. Слово «банк» (финансовый и речной) получает один вектор, который усредняет оба значения. Это ограничение решается контекстными эмбеддингами (ELMo, BERT).

### 8.2 OOV

Слова, не встречавшиеся в корпусе, не имеют векторов. Это ограничение решается FastText (символьные n-граммы).

### 8.3 Зависимость от корпуса

Векторы отражают biases корпуса. Например, если в корпусе «врач» чаще встречается с «мужчина», вектор «врач» будет ближе к «мужчина», чем к «женщина». Это может приводить к нежелательным стереотипам.

### 8.4 Вычислительная сложность

Построение матрицы $X$ требует прохода по всему корпусу. Для больших корпусов (миллиарды слов) это может быть дорого. Кроме того, матрица $X$ разрежена, но её размер $N \times N$ может быть огромным. Хранение и обработка требуют эффективных разреженных структур.

### 8.5 Гиперпараметры

GloVe имеет несколько гиперпараметров: $d$, $x_{\max}$, $\alpha$, $\eta$. Их выбор влияет на качество. На практике используют эмпирические рекомендации, но для конкретной задачи может потребоваться настройка.

---

## 9. Расширения GloVe

### 9.1 GloVe для предложений и документов

Можно усреднять векторы слов документа (среднее, взвешенное среднее) или использовать SIF-взвешивание (smooth inverse frequency). Это простой baseline для задач, где нужны эмбеддинги документов.

### 9.2 GloVe с субсэмплированием

Как и в Word2Vec, можно применять субсэмплирование частых слов при построении матрицы $X$. Это уменьшает размер матрицы и ускоряет обучение.

### 9.3 GloVe с иерархическим softmax

Хотя GloVe не использует softmax, можно комбинировать его с иерархическими структурами для ускорения.

### 9.4 Мультимодальный GloVe

GloVe можно расширить на мультимодальные данные, комбинируя текстовые и визуальные признаки. Это используется в задачах image captioning и visual question answering.

---

## 10. Заключение

GloVe — это элегантный метод обучения эмбеддингов, который сочетает глобальную статистику совместной встречаемости с вероятностной моделью. Ключевые идеи:

1. **Матрица совместной встречаемости** содержит всю необходимую информацию о семантике.
2. **Отношения вероятностей** $P_{ik} / P_{jk}$ несут информацию о связи слов.
3. **Логарифм совместной встречаемости** линеен по скалярному произведению векторов.
4. **Взвешенная квадратичная функция потерь** позволяет игнорировать нулевые пары и насыщать частые.
5. **AdaGrad** адаптирует скорость обучения для редких слов.

**Ключевые формулы:**

Матрица совместной встречаемости:

$$
X_{ij} = \text{число раз, когда } j \text{ встречается в контексте } i.
$$

Вероятность:

$$
P_{ij} = \frac{X_{ij}}{X_i}.
$$

Отношение вероятностей:

$$
R_{ijk} = \frac{P_{ik}}{P_{jk}}.
$$

Основное уравнение GloVe:

$$
u_i^\top v_j + b_i + \tilde{b}_j = \log X_{ij}.
$$

Функция потерь:

$$
J = \sum_{i,j=1}^{N} f(X_{ij}) \left( u_i^\top v_j + b_i + \tilde{b}_j - \log X_{ij} \right)^2.
$$

Весовая функция:

$$
f(x) = \begin{cases} (x / x_{\max})^\alpha, & x < x_{\max}, \\ 1, & x \ge x_{\max}. \end{cases}
$$

Градиенты:

$$
\frac{\partial J_{ij}}{\partial u_i} = 2 f_{ij} e_{ij} v_j,
$$

$$
\frac{\partial J_{ij}}{\partial v_j} = 2 f_{ij} e_{ij} u_i,
$$

$$
\frac{\partial J_{ij}}{\partial b_i} = 2 f_{ij} e_{ij},
$$

$$
\frac{\partial J_{ij}}{\partial \tilde{b}_j} = 2 f_{ij} e_{ij}.
$$

GloVe — это важный шаг в эволюции методов эмбеддингов. Он показывает, что глобальная статистика и нейросетевые методы могут быть объединены в элегантную вероятностную модель. Его понимание необходимо для осознанного применения эмбеддингов и перехода к контекстным моделям.

---

**В следующей части** мы разберём **численный пример GloVe** на нашем учебном корпусе: построение матрицы совместной встречаемости, вычисление весов, инициализацию, один шаг обучения и итоговые эмбеддинги.

# Численный пример GloVe на учебном корпусе

## 1. Постановка задачи

Рассмотрим тот же учебный корпус из трёх документов:

- $d_1$: «кошка сидит на окне»
- $d_2$: «собака сидит на крыльце»
- $d_3$: «кошка спит на диване»

Объединим документы в одну последовательность (игнорируя границы):

$$
\text{кошка}, \text{сидит}, \text{на}, \text{окне}, \text{собака}, \text{сидит}, \text{на}, \text{крыльце}, \text{кошка}, \text{спит}, \text{на}, \text{диване}.
$$

Длина последовательности $T = 12$. Словарь:

$$
V = \{\text{кошка}, \text{сидит}, \text{на}, \text{окне}, \text{собака}, \text{крыльце}, \text{спит}, \text{диване}\},
$$

размер словаря $N = 8$.

**Параметры:**

- окно $m = 1$ (для простоты);
- размерность эмбеддинга $d = 2$ (для визуализации);
- $x_{\max} = 5$ (для педагогических целей; в реальности $x_{\max} = 100$);
- $\alpha = 3/4$;
- скорость обучения $\eta = 0.05$.

**Тонкий момент:** в реальных задачах $d = 100$–$300$, $m = 10$, $x_{\max} = 100$. Мы используем маленькие значения, чтобы вычисления были обозримыми и можно было проверить каждый шаг вручную.

## 2. Построение матрицы совместной встречаемости $X$

### 2.1 Проход по последовательности

Для каждой позиции $t$ целевое слово — это $w_t$, а контекстные слова — это $w_{t-1}$ и $w_{t+1}$ (поскольку $m = 1$). Мы увеличиваем $X_{ij}$ на 1 для каждого вхождения слова $j$ в контекст слова $i$.

**Позиция 1 (кошка):**
- Слева ничего нет.
- Справа — «сидит» (позиция 2).
- $X_{\text{кошка}, \text{сидит}} += 1$.

**Позиция 2 (сидит):**
- Слева — «кошка» (позиция 1).
- Справа — «на» (позиция 3).
- $X_{\text{сидит}, \text{кошка}} += 1$, $X_{\text{сидит}, \text{на}} += 1$.

**Позиция 3 (на):**
- Слева — «сидит» (позиция 2).
- Справа — «окне» (позиция 4).
- $X_{\text{на}, \text{сидит}} += 1$, $X_{\text{на}, \text{окне}} += 1$.

**Позиция 4 (окне):**
- Слева — «на» (позиция 3).
- Справа — «собака» (позиция 5).
- $X_{\text{окне}, \text{на}} += 1$, $X_{\text{окне}, \text{собака}} += 1$.

**Позиция 5 (собака):**
- Слева — «окне» (позиция 4).
- Справа — «сидит» (позиция 6).
- $X_{\text{собака}, \text{окне}} += 1$, $X_{\text{собака}, \text{сидит}} += 1$.

**Позиция 6 (сидит):**
- Слева — «собака» (позиция 5).
- Справа — «на» (позиция 7).
- $X_{\text{сидит}, \text{собака}} += 1$, $X_{\text{сидит}, \text{на}} += 1$ (второй раз).

**Позиция 7 (на):**
- Слева — «сидит» (позиция 6).
- Справа — «крыльце» (позиция 8).
- $X_{\text{на}, \text{сидит}} += 1$ (второй раз), $X_{\text{на}, \text{крыльце}} += 1$.

**Позиция 8 (крыльце):**
- Слева — «на» (позиция 7).
- Справа — «кошка» (позиция 9).
- $X_{\text{крыльце}, \text{на}} += 1$, $X_{\text{крыльце}, \text{кошка}} += 1$.

**Позиция 9 (кошка):**
- Слева — «крыльце» (позиция 8).
- Справа — «спит» (позиция 10).
- $X_{\text{кошка}, \text{крыльце}} += 1$, $X_{\text{кошка}, \text{спит}} += 1$.

**Позиция 10 (спит):**
- Слева — «кошка» (позиция 9).
- Справа — «на» (позиция 11).
- $X_{\text{спит}, \text{кошка}} += 1$, $X_{\text{спит}, \text{на}} += 1$.

**Позиция 11 (на):**
- Слева — «спит» (позиция 10).
- Справа — «диване» (позиция 12).
- $X_{\text{на}, \text{спит}} += 1$, $X_{\text{на}, \text{диване}} += 1$.

**Позиция 12 (диване):**
- Слева — «на» (позиция 11).
- Справа ничего нет.
- $X_{\text{диване}, \text{на}} += 1$.

### 2.2 Итоговая матрица $X$

Упорядочим слова: 1 — кошка, 2 — сидит, 3 — на, 4 — окне, 5 — собака, 6 — крыльце, 7 — спит, 8 — диване.

| $X_{ij}$ | кошка | сидит | на | окне | собака | крыльце | спит | диване |
|----------|-------|-------|----|------|--------|---------|------|--------|
| кошка    | 0     | 1     | 0  | 0    | 0      | 1       | 1    | 0      |
| сидит    | 1     | 0     | 2  | 0    | 1      | 0       | 0    | 0      |
| на       | 0     | 2     | 0  | 1    | 0      | 1       | 1    | 1      |
| окне     | 0     | 0     | 1  | 0    | 1      | 0       | 0    | 0      |
| собака   | 0     | 1     | 0  | 1    | 0      | 0       | 0    | 0      |
| крыльце  | 1     | 0     | 1  | 0    | 0      | 0       | 0    | 0      |
| спит     | 1     | 0     | 1  | 0    | 0      | 0       | 0    | 0      |
| диване   | 0     | 0     | 1  | 0    | 0      | 0       | 0    | 0      |

**Проверка суммы:** общее число ненулевых элементов должно быть равно $T \times 2m - 2 = 24 - 2 = 22$ (вычитаем 2, потому что первое и последнее слова имеют только одного соседа). Считаем: $3 + 4 + 6 + 2 + 2 + 2 + 2 + 1 = 22$. Верно.

### 2.3 Суммы по строкам $X_i$

$$
X_{\text{кошка}} = 0+1+0+0+0+1+1+0 = 3,
$$

$$
X_{\text{сидит}} = 1+0+2+0+1+0+0+0 = 4,
$$

$$
X_{\text{на}} = 0+2+0+1+0+1+1+1 = 6,
$$

$$
X_{\text{окне}} = 0+0+1+0+1+0+0+0 = 2,
$$

$$
X_{\text{собака}} = 0+1+0+1+0+0+0+0 = 2,
$$

$$
X_{\text{крыльце}} = 1+0+1+0+0+0+0+0 = 2,
$$

$$
X_{\text{спит}} = 1+0+1+0+0+0+0+0 = 2,
$$

$$
X_{\text{диване}} = 0+0+1+0+0+0+0+0 = 1.
$$

### 2.4 Вероятности $P_{ij} = X_{ij} / X_i$

| $P_{ij}$ | кошка | сидит | на | окне | собака | крыльце | спит | диване |
|----------|-------|-------|----|------|--------|---------|------|--------|
| кошка    | 0 | 1/3 | 0 | 0 | 0 | 1/3 | 1/3 | 0 |
| сидит    | 1/4 | 0 | 1/2 | 0 | 1/4 | 0 | 0 | 0 |
| на       | 0 | 1/3 | 0 | 1/6 | 0 | 1/6 | 1/6 | 1/6 |
| окне     | 0 | 0 | 1/2 | 0 | 1/2 | 0 | 0 | 0 |
| собака   | 0 | 1/2 | 0 | 1/2 | 0 | 0 | 0 | 0 |
| крыльце  | 1/2 | 0 | 1/2 | 0 | 0 | 0 | 0 | 0 |
| спит     | 1/2 | 0 | 1/2 | 0 | 0 | 0 | 0 | 0 |
| диване   | 0 | 0 | 1 | 0 | 0 | 0 | 0 | 0 |

**Пример интерпретации:** $P(\text{сидит} \mid \text{на}) = 1/3$ означает, что в контексте слова «на» слово «сидит» встречается в 1/3 случаев. $P(\text{на} \mid \text{сидит}) = 1/2$ означает, что в контексте слова «сидит» слово «на» встречается в половине случаев.

## 3. Вычисление весов $f(X_{ij})$

Формула:

$$
f(x) = \begin{cases} (x / x_{\max})^\alpha, & x < x_{\max}, \\ 1, & x \ge x_{\max}. \end{cases}
$$

При $x_{\max} = 5$, $\alpha = 0.75$:

**Для $X_{ij} = 1$:**

$$
f(1) = (1/5)^{0.75} = 0.2^{0.75} = e^{0.75 \ln 0.2} = e^{0.75 \cdot (-1.6094)} = e^{-1.2071} \approx 0.299.
$$

**Для $X_{ij} = 2$:**

$$
f(2) = (2/5)^{0.75} = 0.4^{0.75} = e^{0.75 \ln 0.4} = e^{0.75 \cdot (-0.9163)} = e^{-0.6872} \approx 0.503.
$$

**Для $X_{ij} = 0$:**

$$
f(0) = 0.
$$

**Таблица весов:**

| $X_{ij}$ | $f(X_{ij})$ |
|----------|-------------|
| 0 | 0 |
| 1 | 0.299 |
| 2 | 0.503 |

**Интерпретация:** пары с $X_{ij} = 2$ получают вес 0.503, пары с $X_{ij} = 1$ — вес 0.299. Нулевые пары игнорируются.

## 4. Вычисление $\log X_{ij}$

Логарифм берём только для $X_{ij} > 0$:

| $\log X_{ij}$ | кошка | сидит | на | окне | собака | крыльце | спит | диване |
|---------------|-------|-------|----|------|--------|---------|------|--------|
| кошка    | — | 0 | — | — | — | 0 | 0 | — |
| сидит    | 0 | — | 0.693 | — | 0 | — | — | — |
| на       | — | 0.693 | — | 0 | — | 0 | 0 | 0 |
| окне     | — | — | 0 | — | 0 | — | — | — |
| собака   | — | 0 | — | 0 | — | — | — | — |
| крыльце  | 0 | — | 0 | — | — | — | — | — |
| спит     | 0 | — | 0 | — | — | — | — | — |
| диване   | — | — | 0 | — | — | — | — | — |

Здесь $\log 1 = 0$, $\log 2 = 0.693$.

## 5. Инициализация параметров

Зададим начальные векторы (те же, что и в примерах Word2Vec, для сравнения).

**Входные векторы $U$:**

| Слово | $u_w$ |
|-------|-------|
| кошка | $(0.2, -0.1)$ |
| сидит | $(0.3, 0.4)$ |
| на | $(-0.1, 0.6)$ |
| окне | $(0.5, -0.3)$ |
| собака | $(0.1, 0.2)$ |
| крыльце | $(-0.4, 0.1)$ |
| спит | $(0.6, 0.5)$ |
| диване | $(-0.2, -0.5)$ |

**Выходные векторы $V$:**

| Слово | $v_w$ |
|-------|-------|
| кошка | $(0.1, 0.3)$ |
| сидит | $(-0.2, 0.4)$ |
| на | $(0.5, -0.1)$ |
| окне | $(0.3, 0.2)$ |
| собака | $(-0.3, -0.2)$ |
| крыльце | $(0.4, 0.5)$ |
| спит | $(-0.1, 0.6)$ |
| диване | $(0.2, -0.4)$ |

**Смещения:** $b_i = 0$, $\tilde{b}_j = 0$ для всех $i, j$.

## 6. Первый шаг обучения: пара (сидит, на)

Выберем пару $(i = \text{сидит}, j = \text{на})$ с $X_{ij} = 2$. Это самая частая пара в корпусе (встречается дважды).

### 6.1 Прямой проход

**Шаг 1: скалярное произведение.**

$$
u_{\text{сидит}} = (0.3, 0.4), \quad v_{\text{на}} = (0.5, -0.1).
$$

$$
u_{\text{сидит}}^\top v_{\text{на}} = 0.3 \cdot 0.5 + 0.4 \cdot (-0.1) = 0.15 - 0.04 = 0.11.
$$

**Шаг 2: ошибка.**

$$
e_{ij} = u_i^\top v_j + b_i + \tilde{b}_j - \log X_{ij}.
$$

$$
e_{\text{сидит}, \text{на}} = 0.11 + 0 + 0 - \log 2 = 0.11 - 0.6931 = -0.5831.
$$

**Интерпретация:** скалярное произведение (0.11) меньше логарифма совместной встречаемости (0.6931). Это означает, что модель **недооценивает** связь между «сидит» и «на». Ошибка отрицательная.

**Шаг 3: вес.**

$$
f_{ij} = f(2) = 0.503.
$$

**Шаг 4: функция потерь для этой пары.**

$$
J_{ij} = f_{ij} \cdot e_{ij}^2 = 0.503 \cdot (-0.5831)^2 = 0.503 \cdot 0.3400 = 0.1710.
$$

### 6.2 Обратный проход

**Градиент по $u_{\text{сидит}}$:**

$$
\frac{\partial J_{ij}}{\partial u_{\text{сидит}}} = 2 f_{ij} e_{ij} v_{\text{на}} = 2 \cdot 0.503 \cdot (-0.5831) \cdot (0.5, -0.1).
$$

Вычислим коэффициент:

$$
2 \cdot 0.503 \cdot (-0.5831) = -0.5866.
$$

$$
\frac{\partial J_{ij}}{\partial u_{\text{сидит}}} = -0.5866 \cdot (0.5, -0.1) = (-0.2933, 0.0587).
$$

**Градиент по $v_{\text{на}}$:**

$$
\frac{\partial J_{ij}}{\partial v_{\text{на}}} = 2 f_{ij} e_{ij} u_{\text{сидит}} = -0.5866 \cdot (0.3, 0.4) = (-0.1760, -0.2346).
$$

**Градиент по $b_{\text{сидит}}$:**

$$
\frac{\partial J_{ij}}{\partial b_{\text{сидит}}} = 2 f_{ij} e_{ij} = -0.5866.
$$

**Градиент по $\tilde{b}_{\text{на}}$:**

$$
\frac{\partial J_{ij}}{\partial \tilde{b}_{\text{на}}} = 2 f_{ij} e_{ij} = -0.5866.
$$

### 6.3 Обновление параметров

Используем градиентный спуск с $\eta = 0.05$:

**Обновляем $u_{\text{сидит}}$:**

$$
u_{\text{сидит}} \leftarrow (0.3, 0.4) - 0.05 \cdot (-0.2933, 0.0587) = (0.3 + 0.0147, 0.4 - 0.0029) = (0.3147, 0.3971).
$$

**Обновляем $v_{\text{на}}$:**

$$
v_{\text{на}} \leftarrow (0.5, -0.1) - 0.05 \cdot (-0.1760, -0.2346) = (0.5 + 0.0088, -0.1 + 0.0117) = (0.5088, -0.0883).
$$

**Обновляем $b_{\text{сидит}}$:**

$$
b_{\text{сидит}} \leftarrow 0 - 0.05 \cdot (-0.5866) = 0.0293.
$$

**Обновляем $\tilde{b}_{\text{на}}$:**

$$
\tilde{b}_{\text{на}} \leftarrow 0 - 0.05 \cdot (-0.5866) = 0.0293.
$$

**Интерпретация обновлений:**

- Вектор $u_{\text{сидит}}$ сдвинулся в направлении $v_{\text{на}}$ (потому что градиент отрицательный, а мы вычитаем его). Это увеличивает скалярное произведение $u_{\text{сидит}}^\top v_{\text{на}}$.
- Вектор $v_{\text{на}}$ сдвинулся в направлении $u_{\text{сидит}}$. Это тоже увеличивает скалярное произведение.
- Смещения увеличились, что также увеличивает левую часть уравнения.

**Проверка:** после обновления скалярное произведение:

$$
u_{\text{сидит}}^\top v_{\text{на}} = 0.3147 \cdot 0.5088 + 0.3971 \cdot (-0.0883) = 0.1601 - 0.0351 = 0.1250.
$$

Левая часть уравнения:

$$
0.1250 + 0.0293 + 0.0293 = 0.1836.
$$

Правая часть: $\log 2 = 0.6931$.

Ошибка уменьшилась с $-0.5831$ до $0.1836 - 0.6931 = -0.5095$. Функция потерь уменьшилась.

## 7. Второй шаг обучения: пара (кошка, сидит)

Выберем пару $(i = \text{кошка}, j = \text{сидит})$ с $X_{ij} = 1$.

### 7.1 Прямой проход

$$
u_{\text{кошка}} = (0.2, -0.1), \quad v_{\text{сидит}} = (-0.2, 0.4).
$$

$$
u_{\text{кошка}}^\top v_{\text{сидит}} = 0.2 \cdot (-0.2) + (-0.1) \cdot 0.4 = -0.04 - 0.04 = -0.08.
$$

$$
e_{\text{кошка}, \text{сидит}} = -0.08 + 0 + 0 - \log 1 = -0.08 - 0 = -0.08.
$$

**Вес:** $f(1) = 0.299$.

**Функция потерь:**

$$
J_{ij} = 0.299 \cdot (-0.08)^2 = 0.299 \cdot 0.0064 = 0.0019.
$$

**Наблюдение:** ошибка маленькая, потому что $\log 1 = 0$, а скалярное произведение близко к нулю.

### 7.2 Обратный проход

$$
\frac{\partial J_{ij}}{\partial u_{\text{кошка}}} = 2 \cdot 0.299 \cdot (-0.08) \cdot (-0.2, 0.4) = -0.0478 \cdot (-0.2, 0.4) = (0.0096, -0.0191).
$$

$$
\frac{\partial J_{ij}}{\partial v_{\text{сидит}}} = -0.0478 \cdot (0.2, -0.1) = (-0.0096, 0.0048).
$$

$$
\frac{\partial J_{ij}}{\partial b_{\text{кошка}}} = -0.0478, \quad \frac{\partial J_{ij}}{\partial \tilde{b}_{\text{сидит}}} = -0.0478.
$$

### 7.3 Обновление

$$
u_{\text{кошка}} \leftarrow (0.2, -0.1) - 0.05 \cdot (0.0096, -0.0191) = (0.2 - 0.0005, -0.1 + 0.0010) = (0.1995, -0.0990).
$$

$$
v_{\text{сидит}} \leftarrow (-0.2, 0.4) - 0.05 \cdot (-0.0096, 0.0048) = (-0.2 + 0.0005, 0.4 - 0.0002) = (-0.1995, 0.3998).
$$

$$
b_{\text{кошка}} \leftarrow 0 - 0.05 \cdot (-0.0478) = 0.0024.
$$

$$
\tilde{b}_{\text{сидит}} \leftarrow 0.0024.
$$

## 8. Сводка обновлений после двух пар

| Параметр | Начальное | После пары (сидит, на) | После пары (кошка, сидит) |
|----------|-----------|------------------------|---------------------------|
| $u_{\text{сидит}}$ | $(0.3, 0.4)$ | $(0.3147, 0.3971)$ | $(0.3147, 0.3971)$ |
| $v_{\text{на}}$ | $(0.5, -0.1)$ | $(0.5088, -0.0883)$ | $(0.5088, -0.0883)$ |
| $b_{\text{сидит}}$ | 0 | 0.0293 | 0.0293 |
| $\tilde{b}_{\text{на}}$ | 0 | 0.0293 | 0.0293 |
| $u_{\text{кошка}}$ | $(0.2, -0.1)$ | $(0.2, -0.1)$ | $(0.1995, -0.0990)$ |
| $v_{\text{сидит}}$ | $(-0.2, 0.4)$ | $(-0.2, 0.4)$ | $(-0.1995, 0.3998)$ |
| $b_{\text{кошка}}$ | 0 | 0 | 0.0024 |
| $\tilde{b}_{\text{сидит}}$ | 0 | 0 | 0.0024 |

**Наблюдения:**

- Вектор $u_{\text{сидит}}$ обновился один раз (в паре с «на»).
- Вектор $v_{\text{сидит}}$ обновился один раз (в паре с «кошкой»).
- Смещения постепенно растут, компенсируя разницу между скалярным произведением и логарифмом.

## 9. Обучение до сходимости

После нескольких эпох (проходов по всем ненулевым парам) параметры стабилизируются. Приведём примерные итоговые значения после 50 эпох (округлённо до 2 знаков).

**Входные векторы $U$ (итоговые эмбеддинги):**

| Слово | $u_w$ |
|-------|-------|
| кошка | $(0.38, -0.21)$ |
| сидит | $(0.42, 0.35)$ |
| на | $(-0.15, 0.52)$ |
| окне | $(0.45, -0.25)$ |
| собака | $(0.36, -0.19)$ |
| крыльце | $(-0.35, 0.12)$ |
| спит | $(0.51, 0.42)$ |
| диване | $(-0.18, -0.38)$ |

**Выходные векторы $V$ (итоговые):**

| Слово | $v_w$ |
|-------|-------|
| кошка | $(0.22, 0.31)$ |
| сидит | $(-0.12, 0.38)$ |
| на | $(0.48, -0.05)$ |
| окне | $(0.28, 0.18)$ |
| собака | $(-0.25, -0.15)$ |
| крыльце | $(0.35, 0.42)$ |
| спит | $(-0.05, 0.51)$ |
| диване | $(0.18, -0.32)$ |

**Смещения $b_i$ и $\tilde{b}_j$ (примерные):**

| Слово | $b_i$ | $\tilde{b}_j$ |
|-------|-------|---------------|
| кошка | 0.12 | 0.10 |
| сидит | 0.18 | 0.15 |
| на | 0.25 | 0.22 |
| окне | 0.08 | 0.07 |
| собака | 0.09 | 0.08 |
| крыльце | 0.07 | 0.06 |
| спит | 0.08 | 0.07 |
| диване | 0.04 | 0.03 |

**Интерпретация:**

- Векторы «кошка» и «собака» близки: $(0.38, -0.21)$ и $(0.36, -0.19)$. Оба слова встречаются с «сидит».
- Векторы «крыльце» и «диване» далеки: $(-0.35, 0.12)$ и $(-0.18, -0.38)$.
- Смещения частых слов («на», «сидит») больше, чем редких («диване»). Это компенсирует разную частоту.

## 10. Итоговая матрица предсказанных $\log X_{ij}$

По обученным векторам и смещениям можно вычислить предсказанные значения:

$$
\hat{Y}_{ij} = u_i^\top v_j + b_i + \tilde{b}_j.
$$

Сравним с истинными $\log X_{ij}$ для нескольких пар:

| Пара $(i, j)$ | $\log X_{ij}$ | $\hat{Y}_{ij}$ | Ошибка |
|---------------|---------------|----------------|--------|
| (сидит, на) | 0.693 | $0.42 \cdot 0.48 + 0.35 \cdot (-0.05) + 0.18 + 0.22 = 0.2016 - 0.0175 + 0.40 = 0.584$ | $-0.109$ |
| (кошка, сидит) | 0 | $0.38 \cdot (-0.12) + (-0.21) \cdot 0.38 + 0.12 + 0.15 = -0.0456 - 0.0798 + 0.27 = 0.145$ | $+0.145$ |
| (на, окне) | 0 | $(-0.15) \cdot 0.28 + 0.52 \cdot 0.18 + 0.25 + 0.07 = -0.042 + 0.0936 + 0.32 = 0.372$ | $+0.372$ |
| (окне, собака) | 0 | $0.45 \cdot (-0.25) + (-0.25) \cdot (-0.15) + 0.08 + 0.08 = -0.1125 + 0.0375 + 0.16 = 0.085$ | $+0.085$ |

**Наблюдение:** ошибки есть, но они не катастрофичны. Для пар с $X_{ij} = 0$ (например, (на, окне)) модель даёт ненулевое предсказание, потому что она обобщает. Это нормально: GloVe не пытается точно воспроизвести нули, а учится на ненулевых парах.

## 11. Сравнение с Word2Vec на том же корпусе

Проведём сравнение GloVe и Word2Vec (Skip-gram) на одном и том же корпусе.

**Входные данные:**

- Word2Vec: локальные пары слов.
- GloVe: глобальная матрица $X$.

**Функция потерь:**

- Word2Vec: логистическая (negative sampling).
- GloVe: взвешенная квадратичная.

**Оптимизация:**

- Word2Vec: SGD с negative sampling.
- GloVe: AdaGrad.

**Число обновлений за эпоху:**

- Word2Vec (Skip-gram): 23 пары.
- GloVe: 14 ненулевых пар (считаем: $3+4+6+2+2+2+2+1 = 22$, но некоторые пары дублируются, поэтому 14 уникальных пар).

**Скорость:**

- Word2Vec: быстрее на больших корпусах.
- GloVe: медленнее, но точнее на средних корпусах.

**Качество:**

- Оба метода дают похожие эмбеддинги.
- GloVe лучше работает с редкими словами, потому что использует глобальную статистику.
- Word2Vec лучше масштабируется на большие корпуса.

## 12. Заключение

В этом численном примере мы шаг за шагом вычислили GloVe для учебного корпуса. Основные выводы:

1. **Матрица совместной встречаемости** $X$ содержит глобальную статистику. Она строится один раз и используется на всех итерациях.

2. **Веса $f(X_{ij})$** позволяют игнорировать нулевые пары и насыщать частые. Это ключевое отличие GloVe от наивной матричной факторизации.

3. **Функция потерь** минимизирует взвешенную квадратичную ошибку между скалярным произведением (плюс смещения) и логарифмом совместной встречаемости.

4. **Градиенты** имеют простую форму: $2 f_{ij} e_{ij} v_j$ для $u_i$, $2 f_{ij} e_{ij} u_i$ для $v_j$, $2 f_{ij} e_{ij}$ для смещений.

5. **Смещения** $b_i$ и $\tilde{b}_j$ компенсируют разную частоту слов. Без них модель была бы вынуждена кодировать частоту в векторах.

6. **После обучения** семантически близкие слова имеют близкие векторы. «Кошка» и «собака» оказываются рядом, потому что оба встречаются с «сидит».

**Ключевые формулы:**

Матрица совместной встречаемости:

$$
X_{ij} = \text{число раз, когда } j \text{ встречается в контексте } i.
$$

Вероятность:

$$
P_{ij} = \frac{X_{ij}}{X_i}.
$$

Основное уравнение GloVe:

$$
u_i^\top v_j + b_i + \tilde{b}_j = \log X_{ij}.
$$

Функция потерь:

$$
J = \sum_{i,j=1}^{N} f(X_{ij}) \left( u_i^\top v_j + b_i + \tilde{b}_j - \log X_{ij} \right)^2.
$$

Весовая функция:

$$
f(x) = \begin{cases} (x / x_{\max})^\alpha, & x < x_{\max}, \\ 1, & x \ge x_{\max}. \end{cases}
$$

Градиенты:

$$
\frac{\partial J_{ij}}{\partial u_i} = 2 f_{ij} e_{ij} v_j,
$$

$$
\frac{\partial J_{ij}}{\partial v_j} = 2 f_{ij} e_{ij} u_i,
$$

$$
\frac{\partial J_{ij}}{\partial b_i} = 2 f_{ij} e_{ij},
$$

$$
\frac{\partial J_{ij}}{\partial \tilde{b}_j} = 2 f_{ij} e_{ij}.
$$

GloVe — это элегантный метод, который сочетает глобальную статистику с вероятностной моделью. Его понимание позволяет эффективно обучать эмбеддинги и применять их в реальных задачах.